# Figure 3 (aggregated across runs) — variant-assignment comparison

Across-runs version of `manuscript-figure-3-variant-assignment-compare.ipynb`.
Reads the small committed tables written by `scripts/aggregate_results.py`
(under `results/aggregated/<batch>/`) and shows, per assignment method,
each sweep config as a mean±band over its replicate runs.

Tables consumed:
- `variant_counts_over_time.csv` — `batch, config, run, year_bin, method, n_variants`
- `method_agreement_nid.csv` — `batch, config, run, method_x, method_y, nid`
- `fitness_variance_over_time.csv` *(optional)* — `batch, config, run, year, method, mean_variance, n_variants`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-paper')

# --- parameters ---------------------------------------------------------
BATCH = '2026-07-04-reviewer-runs'          # sweep batch name
AGG_DIR = Path('..') / 'results' / 'aggregated' / BATCH
FIG_DIR = Path('..') / 'figures'            # where to save (created on demand)
TRUE_N_VARIANTS = 3                          # dashed reference line; confirm/parameterize
SPREAD = 'ci95'                             # 'ci95' | 'sd' band around the mean

METHODS = ['variant_ag', 'variant_tsne', 'variant_phylo']
METHOD_LABELS = {
    'variant_ag': 'antigenic',
    'variant_tsne': 'sequence',
    'variant_phylo': 'phylogenetic',
}
METHOD_COLORS = {
    'variant_ag': '#1f77b4',
    'variant_tsne': '#ff7f0e',
    'variant_phylo': '#2ca02c',
}

In [ ]:
# --- load the aggregated tables ----------------------------------------
counts_df = pd.read_csv(AGG_DIR / 'variant_counts_over_time.csv')
nid_df = pd.read_csv(AGG_DIR / 'method_agreement_nid.csv')

variance_path = AGG_DIR / 'fitness_variance_over_time.csv'
variance_df = pd.read_csv(variance_path) if variance_path.exists() else None
if variance_df is None:
    print('fitness_variance_over_time.csv not found; variance panel skipped')

configs = sorted(counts_df['config'].unique())
config_palette = dict(zip(configs, sns.color_palette('husl', len(configs))))
print(f'{len(configs)} configs, {counts_df["run"].nunique()} distinct run ids')

In [ ]:
def config_band(df, value_col, x_col):
    """Return per-(config, x) mean and lo/hi band across replicate runs."""
    grouped = df.groupby(['config', x_col])[value_col]
    agg = grouped.agg(['mean', 'std', 'count']).reset_index()
    if SPREAD == 'sd':
        half = agg['std'].fillna(0.0)
    else:  # 95% CI from the standard error of the mean.
        sem = agg['std'].fillna(0.0) / np.sqrt(agg['count'].clip(lower=1))
        half = 1.96 * sem
    agg['lo'] = agg['mean'] - half
    agg['hi'] = agg['mean'] + half
    return agg


def plot_over_time(source_df, value_col, x_col, ylabel, ref_line=None):
    """One panel per method; each config drawn as a mean±band line."""
    methods = [m for m in METHODS if m in source_df['method'].unique()]
    fig, axes = plt.subplots(
        1, len(methods), figsize=(5 * len(methods), 4), sharex=True, sharey=True
    )
    axes = np.atleast_1d(axes)
    for ax, method in zip(axes, methods):
        sub = source_df[source_df['method'] == method]
        for config in configs:
            band = config_band(sub[sub['config'] == config], value_col, x_col)
            if band.empty:
                continue
            color = config_palette[config]
            ax.plot(band[x_col], band['mean'], color=color, label=config, lw=1.5)
            ax.fill_between(band[x_col], band['lo'], band['hi'], color=color, alpha=0.18)
        if ref_line is not None:
            ax.axhline(ref_line, ls='--', color='red', lw=1, zorder=0)
        ax.set_title(METHOD_LABELS.get(method, method))
        ax.set_xlabel('year')
        sns.despine(ax=ax)
    axes[0].set_ylabel(ylabel)
    axes[-1].legend(title='config', fontsize=7, frameon=False)
    fig.tight_layout()
    return fig

In [ ]:
# --- Panel row 1: number of variants over time -------------------------
fig_counts = plot_over_time(
    counts_df, 'n_variants', 'year_bin',
    ylabel='number of variants', ref_line=TRUE_N_VARIANTS,
)
fig_counts.suptitle('Variants per method over time (mean ± band across replicates)', y=1.02)

In [ ]:
# --- Panel row 2 (optional): within-variant fitness variance over time --
if variance_df is not None:
    fig_var = plot_over_time(
        variance_df, 'mean_variance', 'year',
        ylabel='mean within-variant fitness variance',
    )
    fig_var.suptitle('Fitness variance per method over time', y=1.02)
else:
    fig_var = None

In [ ]:
# --- Panel row 3: method-agreement (NID) distributions across replicates -
short = {'variant_ag': 'ag', 'variant_tsne': 'seq', 'variant_phylo': 'phylo'}
nid_df = nid_df.copy()
nid_df['pair'] = (
    nid_df['method_x'].map(short).fillna(nid_df['method_x'])
    + ' vs '
    + nid_df['method_y'].map(short).fillna(nid_df['method_y'])
)
fig_nid, ax = plt.subplots(figsize=(1.6 * max(len(configs), 3) + 2, 4))
sns.boxplot(data=nid_df, x='config', y='nid', hue='pair', ax=ax, fliersize=0)
sns.stripplot(
    data=nid_df, x='config', y='nid', hue='pair',
    ax=ax, dodge=True, size=3, alpha=0.5, legend=False,
)
ax.set_ylabel('NID (0 = identical, 1 = independent)')
ax.set_xlabel('config')
ax.legend(title='method pair', fontsize=7, frameon=False)
sns.despine(ax=ax)
fig_nid.tight_layout()

In [ ]:
# --- save (pdf + png, dpi=300) -----------------------------------------
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    if fig is None:
        return
    for ext in ('pdf', 'png'):
        fig.savefig(FIG_DIR / f'{name}.{ext}', dpi=300, bbox_inches='tight')

save(fig_counts, f'figure3_aggregated_variant_counts_{BATCH}')
save(fig_var, f'figure3_aggregated_fitness_variance_{BATCH}')
save(fig_nid, f'figure3_aggregated_nid_{BATCH}')
print('saved to', FIG_DIR.resolve())